Thực hành dò biên:
1. Thực hành các phép phát hiện biên trực tiếp trên ảnh ví dụ 5x5 và 1 ảnh xám văn bản (Hiển thị như nào là tốt nhất)
3. Tự chọn 2-3 ảnh biển số xe. Sử dụng kết hợp phát hiện biên trực tiếp, phân ngưỡng và tìm vùng liên thông để tách riêng phần biển số (và có thể cả chữ số) trên các ảnh đó


In [1]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

# M, N: kích thước ảnh (rộng, cao)
# m, n: kích thước mẫu (rộng, cao)
# p: điểm chốt
# x, y: điểm được kiểm tra xem có thể đặt mẫu theo chốt không
def co_the_tinh(x, y, M, N, m, n, p):
    x_min = x - p[0]
    y_min = y - p[1]
    x_max = x + m - p[0] - 1
    y_max = y + n - p[1] - 1
    return x_min >= 0 and y_min >= 0 and x_max < M and y_max < N

# p: điểm chốt, mặc định tại (0, 0) - điểm trên cùng bên trái trong mẫu
# inv: giá trị không tính được (thường ký hiệu là *), mặc định giá trị vô cực
def phep_cuon_2(I, T, p=(0, 0), inv=np.inf):
    M, N = I.shape[1], I.shape[0]
    m, n = T.shape[1], T.shape[0]
    I_kq = np.zeros((N, M), np.float32)
    for x in range(I_kq.shape[1]):
        for y in range(I_kq.shape[0]):
            if co_the_tinh(x, y, M, N, m, n, p):
                sum_val = 0.0
                for j in range(n):
                    for i in range(m):
                        sum_val = sum_val + I[y + j - p[1], x + i - p[0]] * T[j, i]
                I_kq[y, x] = sum_val
            else:
                I_kq[y, x] = inv
    return I_kq

## 1. Phát hiện biên trực tiếp trên ảnh 5x5 và ảnh xám văn bản 

In [2]:
# 1. Phép Gradient (Mẫu 1x2 và 2x1, chốt tại (0,0))
# Mẫu 1x2 -> dò sự thay đổi theo phương ngang 
# Mẫu 2x1 dò sự thay đổi theo phương dọc 
def phep_gradient(I):
    A = np.array([[-1, 1]], dtype=np.float32)
    B = np.array([[-1], [1]], dtype=np.float32)
    IxA = phep_cuon_2(I, A)
    IxB = phep_cuon_2(I, B)
    I_kq = IxA + IxB
    return I_kq, IxA, IxB

# 2. Phép Robert (Mẫu chéo 2x2, chốt tại (0,0))
# Đo độ lệch theo 2 phương đường chéo 
# IxHy = I(y + 1, x) - I(y, x + 1) 
def phep_robert(I):
    Hx = np.array([
        [-1, 0],
        [ 0, 1]
    ], dtype=np.float32)
    Hy = np.array([
        [ 0, -1],
        [ 1,  0]
    ], dtype=np.float32)
    IxHx = phep_cuon_2(I, Hx)
    IxHy = phep_cuon_2(I, Hy)
    I_kq = IxHx + IxHy
    return I_kq, IxHx, IxHy

# 3. Phép Prewitt (Mẫu 3x3, đặt tâm tại (1,1) để biên cân đối)
def phep_prewitt(I, p=(1, 1)):
    Hx = np.array([
        [-1, 0, 1],
        [-1, 0, 1],
        [-1, 0, 1],
    ], dtype=np.float32)
    Hy = np.array([
        [-1, -1, -1],
        [ 0,  0,  0],
        [ 1,  1,  1],
    ], dtype=np.float32)
    IxHx = phep_cuon_2(I, Hx, p=p)
    IxHy = phep_cuon_2(I, Hy, p=p)
    I_kq = IxHx + IxHy
    return I_kq, IxHx, IxHy

# 4. Phép Sobel (Mẫu 3x3 nhấn mạnh trọng số ở tâm, chốt tại (1,1))
def phep_sobel(I, p=(1, 1)):
    Hx = np.array([
        [-1, 0, 1],
        [-2, 0, 2],
        [-1, 0, 1],
    ], dtype=np.float32)
    Hy = np.array([
        [-1, -2, -1],
        [ 0,  0,  0],
        [ 1,  2,  1],
    ], dtype=np.float32)
    IxHx = phep_cuon_2(I, Hx, p=p)
    IxHy = phep_cuon_2(I, Hy, p=p)
    I_kq = IxHx + IxHy
    return I_kq, IxHx, IxHy

In [3]:
I_5x5 = np.array([
    [0, 0, 0, 0, 0],
    [0, 8, 8, 8, 0],
    [0, 8, 8, 8, 0],
    [0, 8, 8, 8, 0],
    [0, 0, 0, 0, 0]
], dtype=np.uint8)

print("=== ẢNH BAN ĐẦU 5x5 ===")
print(I_5x5)

for ten_phep, ham_do_bien in [
    ("Gradient", phep_gradient),
    ("Robert", phep_robert),
    ("Prewitt (chốt 0,0)", lambda img: phep_prewitt(img, p=(0,0))),
    ("Sobel (chốt 1,1)", lambda img: phep_sobel(img, p=(1,1)))
]:
    I_kq, IxHx, IxHy = ham_do_bien(I_5x5)
    print(f"\n--- Phép {ten_phep} ---")
    print("I cuộn Hx:\n", IxHx)
    print("I cuộn Hy:\n", IxHy)
    print("Ảnh kết quả (I_kq):\n", I_kq)

=== ẢNH BAN ĐẦU 5x5 ===
[[0 0 0 0 0]
 [0 8 8 8 0]
 [0 8 8 8 0]
 [0 8 8 8 0]
 [0 0 0 0 0]]

--- Phép Gradient ---
I cuộn Hx:
 [[ 0.  0.  0.  0. inf]
 [ 8.  0.  0. -8. inf]
 [ 8.  0.  0. -8. inf]
 [ 8.  0.  0. -8. inf]
 [ 0.  0.  0.  0. inf]]
I cuộn Hy:
 [[ 0.  8.  8.  8.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0.  0.  0.  0.  0.]
 [ 0. -8. -8. -8.  0.]
 [inf inf inf inf inf]]
Ảnh kết quả (I_kq):
 [[  0.   8.   8.   8.  inf]
 [  8.   0.   0.  -8.  inf]
 [  8.   0.   0.  -8.  inf]
 [  8.  -8.  -8. -16.  inf]
 [ inf  inf  inf  inf  inf]]

--- Phép Robert ---
I cuộn Hx:
 [[ 8.  8.  8.  0. inf]
 [ 8.  0.  0. -8. inf]
 [ 8.  0.  0. -8. inf]
 [ 0. -8. -8. -8. inf]
 [inf inf inf inf inf]]
I cuộn Hy:
 [[ 0.  8.  8.  8. inf]
 [-8.  0.  0.  8. inf]
 [-8.  0.  0.  8. inf]
 [-8. -8. -8.  0. inf]
 [inf inf inf inf inf]]
Ảnh kết quả (I_kq):
 [[  8.  16.  16.   8.  inf]
 [  0.   0.   0.   0.  inf]
 [  0.   0.   0.   0.  inf]
 [ -8. -16. -16.  -8.  inf]
 [ inf  inf  inf  inf  inf]]

--- Phép Prewitt (chốt 0,0) -

In [4]:
from collections import deque

# Hàm phân ngưỡng mức xám cơ bản
def phan_nguong_muc_xam(I, theta, lay_vung_toi=True):
    I_kq = np.zeros(I.shape, np.uint8)
    for i in range(I.shape[0]):
        for j in range(I.shape[1]):
            if lay_vung_toi:
                I_kq[i, j] = 255 if I[i, j] < theta else 0
            else:
                I_kq[i, j] = 255 if I[i, j] > theta else 0
    return I_kq

# Cải tiến hàm tìm vùng liên thông: Hỗ trợ bán kính ngang (bk_x) và dọc (bk_y) riêng biệt
# Giúp nối các chữ số nằm ngang sát nhau trên biển số mà không bị nhảy dọc lên lưới tản nhiệt/cản xe
def tim_vung_lien_thong(I_nhi_phan, bk_x=1, bk_y=1):
    N, M = I_nhi_phan.shape
    da_tham = np.zeros((N, M), dtype=bool)
    danh_sach_vung = []

    for y in range(N):
        for x in range(M):
            if I_nhi_phan[y, x] == 255 and not da_tham[y, x]:
                hang_doi = deque([(x, y)])
                da_tham[y, x] = True
                x_min, x_max = x, x
                y_min, y_max = y, y
                so_diem = 0

                while hang_doi:
                    cx, cy = hang_doi.popleft()
                    so_diem += 1
                    if cx < x_min: x_min = cx
                    if cx > x_max: x_max = cx
                    if cy < y_min: y_min = cy
                    if cy > y_max: y_max = cy

                    # Loang theo bk_x (ngang) và bk_y (dọc)
                    for ny in range(max(0, cy - bk_y), min(N, cy + bk_y + 1)):
                        for nx in range(max(0, cx - bk_x), min(M, cx + bk_x + 1)):
                            if I_nhi_phan[ny, nx] == 255 and not da_tham[ny, nx]:
                                da_tham[ny, nx] = True
                                hang_doi.append((nx, ny))

                rong = x_max - x_min + 1
                cao = y_max - y_min + 1
                danh_sach_vung.append({
                    'bbox': (x_min, y_min, x_max, y_max),
                    'rong': rong,
                    'cao': cao,
                    'dien_tich_hop': rong * cao,
                    'so_diem': so_diem
                })
    return danh_sach_vung


def tach_bien_so_va_ky_tu_thuc_te(duong_dan_anh):
    I_goc = cv2.imread(duong_dan_anh, cv2.IMREAD_GRAYSCALE)
    if I_goc is None:
        print(f"Không đọc được file {duong_dan_anh}")
        return

    # Thu nhỏ ảnh về chiều rộng 400px để chạy phép cuộn tìm vị trí biển số cho nhanh
    N_goc, M_goc = I_goc.shape
    M_moi = 400
    N_moi = int(N_goc * (400.0 / M_goc))
    I_xu_ly = cv2.resize(I_goc, (M_moi, N_moi))
    N, M = I_xu_ly.shape

    # BƯỚC 1: Chỉ dùng BIÊN DỌC (IxHx) của Prewitt để bắt các nét chữ thẳng đứng trên biển số
    # Loại bỏ hoàn toàn các đường kẻ ngang của cản xe, nắp cốp, lưới tản nhiệt
    _, IxHx, _ = phep_prewitt(I_xu_ly, p=(1, 1))
    IxHx_sach = np.where(np.isinf(IxHx), 0, np.abs(IxHx))

    # Phân ngưỡng lấy 10% điểm có biên dọc mạnh nhất
    nguong_bien = np.percentile(IxHx_sach, 90)
    I_nhi_phan_bien = sinh_anh_hien_thi_bien(IxHx_sach, theta=nguong_bien, nen_trang=False)

    # BƯỚC 2: Tìm vùng liên thông (nối ngang bk_x=4 để gom các chữ số, nối dọc bk_y=1 để tách khỏi xe)
    cac_vung = tim_vung_lien_thong(I_nhi_phan_bien, bk_x=4, bk_y=1)

    # BƯỚC 3: Lọc và chấm điểm vùng biển số
    vung_bien_so = None
    diem_tot_nhat = -1

    for vung in cac_vung:
        x1, y1, x2, y2 = vung['bbox']
        w, h = vung['rong'], vung['cao']
        ty_le = w / float(h)
        dien_tich = vung['dien_tich_hop']
        mat_do_bien = vung['so_diem'] / float(dien_tich)

        # Điều kiện kích thước thực tế của biển số trong ảnh chụp xe:
        # - Diện tích chỉ từ 0.2% đến 6% diện tích ảnh (loại bỏ các khung cây cối/thân xe khổng lồ)
        # - Tỷ lệ Rộng/Cao từ 1.0 (biển vuông 2 dòng) đến 5.5 (biển dài 1 dòng)
        # - Vị trí nằm ở phần thân xe (không nằm ở 25% mép trên cùng của ảnh)
        if (0.002 * M * N <= dien_tich <= 0.06 * M * N) and \
           (1.0 <= ty_le <= 5.5) and \
           (0.025 * N <= h <= 0.22 * N) and \
           (y1 > 0.25 * N):
            
            # Đặc trưng vật lý: Vùng biển số có nền trắng sáng và độ tương phản (std) rất cao
            vung_anh_xam = I_xu_ly[y1:y2+1, x1:x2+1]
            do_sang_nen = np.percentile(vung_anh_xam, 80)
            do_tuong_phan = np.std(vung_anh_xam)
            
            # Điểm đánh giá kết hợp mật độ biên dọc, độ tương phản và độ sáng nền biển
            diem = mat_do_bien * do_tuong_phan * do_sang_nen
            if diem > diem_tot_nhat:
                diem_tot_nhat = diem
                vung_bien_so = vung

    if vung_bien_so is None:
        print(f"[{duong_dan_anh}] Không tìm thấy vùng biển số!")
        return

    # Ánh xạ tọa độ biển số từ ảnh thu nhỏ (I_xu_ly) về ẢNH GỐC ĐỘ PHÂN GIẢI CAO (I_goc)
    x1, y1, x2, y2 = vung_bien_so['bbox']
    ty_le_x = M_goc / float(M)
    ty_le_y = N_goc / float(N)
    
    # Mở rộng nhẹ khung cắt 2 pixel để lấy trọn vẹn biển số trên ảnh gốc
    gx1 = max(0, int((x1 - 2) * ty_le_x))
    gy1 = max(0, int((y1 - 2) * ty_le_y))
    gx2 = min(M_goc - 1, int((x2 + 2) * ty_le_x))
    gy2 = min(N_goc - 1, int((y2 + 2) * ty_le_y))
    
    I_bien_so_goc = I_goc[gy1:gy2+1, gx1:gx2+1]

    # Phóng chuẩn hóa chiều cao biển số về 100 pixel để tách chữ số rõ nét nhất
    cao_bs, rong_bs = I_bien_so_goc.shape
    rong_chuan = int(rong_bs * (100.0 / cao_bs))
    I_bien_so = cv2.resize(I_bien_so_goc, (rong_chuan, 100))

    # BƯỚC 4: Phân ngưỡng tách chữ số trên ảnh biển số đã chuẩn hóa
    # Lấy ngưỡng theo vùng trung tâm của biển số để không bị ảnh hưởng bởi viền đen bên ngoài
    vung_tam_bs = I_bien_so[15:85, 10:max(11, rong_chuan-10)]
    nguong_chu = np.percentile(vung_tam_bs, 40)
    I_nhi_phan_chu = phan_nguong_muc_xam(I_bien_so, theta=nguong_chu, lay_vung_toi=True)

    # Xóa 4 pixel sát rìa ngoài để chữ số không bị nối dính vào khung viền đen của biển số
    I_nhi_phan_chu[:4, :] = 0; I_nhi_phan_chu[-4:, :] = 0
    I_nhi_phan_chu[:, :4] = 0; I_nhi_phan_chu[:, -4:] = 0

    # Tìm vùng liên thông cho từng ký tự (bk_x=1, bk_y=1)
    cac_vung_chu = tim_vung_lien_thong(I_nhi_phan_chu, bk_x=1, bk_y=1)
    nb_cao, nb_rong = I_bien_so.shape

    danh_sach_ky_tu = []
    for vung in cac_vung_chu:
        w, h = vung['rong'], vung['cao']
        ty_le_chu = w / float(h)
        # Lọc kích thước chữ cái/chữ số (hỗ trợ cả biển 1 dòng và biển 2 dòng)
        if (0.22 * nb_cao <= h <= 0.88 * nb_cao) and \
           (0.03 * nb_rong <= w <= 0.30 * nb_rong) and \
           (0.12 <= ty_le_chu <= 1.05):
            danh_sach_ky_tu.append(vung['bbox'])

    # Sắp xếp thứ tự ký tự từ trên xuống dưới, từ trái sang phải
    danh_sach_ky_tu.sort(key=lambda b: (b[1] // (nb_cao // 2), b[0]))

    # Hiển thị kết quả 4 bước
    I_hien_thi_bs = cv2.cvtColor(I_xu_ly, cv2.COLOR_GRAY2RGB)
    cv2.rectangle(I_hien_thi_bs, (x1, y1), (x2, y2), (0, 255, 0), 2)

    I_hien_thi_ky_tu = cv2.cvtColor(I_bien_so, cv2.COLOR_GRAY2RGB)
    for (kx1, ky1, kx2, ky2) in danh_sach_ky_tu:
        cv2.rectangle(I_hien_thi_ky_tu, (kx1, ky1), (kx2, ky2), (255, 0, 0), 2)

    plt.figure(figsize=(16, 4))
    plt.suptitle(f"File: {duong_dan_anh} | Tách được {len(danh_sach_ky_tu)} vùng ký tự", fontsize=13, fontweight='bold')
    plt.subplot(1, 4, 1); plt.imshow(I_xu_ly, cmap='gray'); plt.title("1. Ảnh xám gốc"); plt.axis("off")
    plt.subplot(1, 4, 2); plt.imshow(I_nhi_phan_bien, cmap='gray'); plt.title("2. Biên dọc Prewitt + Phân ngưỡng"); plt.axis("off")
    plt.subplot(1, 4, 3); plt.imshow(I_hien_thi_bs); plt.title("3. Khoanh vùng liên thông Biển số"); plt.axis("off")
    plt.subplot(1, 4, 4); plt.imshow(I_hien_thi_ky_tu); plt.title("4. Cắt biển số & Tách chữ số"); plt.axis("off")
    plt.tight_layout()
    plt.show()

# Chạy lại trên 3 file ảnh của bạn
danh_sach_file_bsx = ["bsx1.jpg", "bsx2.jpeg", "bsx3.jpg"]
for ten_file in danh_sach_file_bsx:
    tach_bien_so_va_ky_tu_thuc_te(ten_file)

NameError: name 'sinh_anh_hien_thi_bien' is not defined

## Tách biển số xe và chữ số kết hợp Dò biên, phân ngưỡng và tìm vùng liên thông 

In [ ]:
from collections import deque

# Hàm phân ngưỡng mức xám cơ bản (dùng để tách chữ đen trên nền biển số trắng)
def phan_nguong_muc_xam(I, theta, lay_vung_toi=True):
    I_kq = np.zeros(I.shape, np.uint8)
    for i in range(I.shape[0]):
        for j in range(I.shape[1]):
            if lay_vung_toi:
                I_kq[i, j] = 255 if I[i, j] < theta else 0
            else:
                I_kq[i, j] = 255 if I[i, j] > theta else 0
    return I_kq

# Thuật toán tìm các vùng liên thông bằng duyệt chiều rộng (BFS)
# ban_kinh: khoảng cách tối đa giữa 2 điểm trắng để coi là cùng 1 vùng liên thông
def tim_vung_lien_thong(I_nhi_phan, ban_kinh=1):
    N, M = I_nhi_phan.shape
    da_tham = np.zeros((N, M), dtype=bool)
    danh_sach_vung = []

    for y in range(N):
        for x in range(M):
            if I_nhi_phan[y, x] == 255 and not da_tham[y, x]:
                # Khởi tạo vùng liên thông mới
                hang_doi = deque([(x, y)])
                da_tham[y, x] = True
                x_min, x_max = x, x
                y_min, y_max = y, y
                so_diem = 0

                while hang_doi:
                    cx, cy = hang_doi.popleft()
                    so_diem += 1
                    if cx < x_min: x_min = cx
                    if cx > x_max: x_max = cx
                    if cy < y_min: y_min = cy
                    if cy > y_max: y_max = cy

                    # Duyệt các điểm lân cận trong phạm vi ban_kinh
                    for ny in range(max(0, cy - ban_kinh), min(N, cy + ban_kinh + 1)):
                        for nx in range(max(0, cx - ban_kinh), min(M, cx + ban_kinh + 1)):
                            if I_nhi_phan[ny, nx] == 255 and not da_tham[ny, nx]:
                                da_tham[ny, nx] = True
                                hang_doi.append((nx, ny))

                rong = x_max - x_min + 1
                cao = y_max - y_min + 1
                danh_sach_vung.append({
                    'bbox': (x_min, y_min, x_max, y_max),
                    'rong': rong,
                    'cao': cao,
                    'dien_tich_hop': rong * cao,
                    'so_diem': so_diem
                })
    return danh_sach_vung

In [ ]:
def tach_bien_so_va_ky_tu_thuc_te(duong_dan_anh, ban_kinh_gom_bien=4):
    I_goc = cv2.imread(duong_dan_anh, cv2.IMREAD_GRAYSCALE)
    if I_goc is None:
        print(f"Không đọc được file {duong_dan_anh}")
        return

    # Chuẩn hóa chiều rộng về khoảng 400px để phep_cuon_2 (thuần vòng lặp Python) chạy nhanh
    N_goc, M_goc = I_goc.shape
    if M_goc > 400:
        M_moi = 400
        N_moi = int(N_goc * (400.0 / M_goc))
        I_xu_ly = cv2.resize(I_goc, (M_moi, N_moi))
    else:
        I_xu_ly = I_goc.copy()
        
    N, M = I_xu_ly.shape

    # BƯỚC 1: Phát hiện biên bằng Prewitt và phân ngưỡng tự động (lấy top 12% điểm có biên mạnh nhất)
    _, IxHx, IxHy = phep_prewitt(I_xu_ly, p=(1, 1))
    IxHx_sach = np.where(np.isinf(IxHx), 0, np.abs(IxHx))
    IxHy_sach = np.where(np.isinf(IxHy), 0, np.abs(IxHy))
    
    # Biển số xe có đặc trưng biên dọc (từ các chữ số) rất mạnh kết hợp với biên ngang của khung
    I_bien_tong = IxHx_sach + 0.5 * IxHy_sach
    nguong_bien = np.percentile(I_bien_tong, 88)
    I_nhi_phan_bien = sinh_anh_hien_thi_bien(I_bien_tong, theta=nguong_bien, nen_trang=False)

    # BƯỚC 2: Tìm vùng liên thông trên ảnh biên với bán kính lân cận để gom cụm biển số
    cac_vung = tim_vung_lien_thong(I_nhi_phan_bien, ban_kinh=ban_kinh_gom_bien)

    # BƯỚC 3: Lọc vùng liên thông ứng cử viên là biển số xe
    vung_bien_so = None
    diem_tot_nhat = -1

    for vung in cac_vung:
        w, h = vung['rong'], vung['cao']
        ty_le = w / float(h)
        dien_tich = vung['dien_tich_hop']
        mat_do = vung['so_diem'] / float(dien_tich)

        # Biển số xe có tỷ lệ Rộng/Cao từ 1.1 (biển vuông 2 dòng) đến 5.5 (biển dài 1 dòng)
        if 1.1 <= ty_le <= 5.5 and (0.015 * M * N) <= dien_tich <= (0.85 * M * N) and mat_do >= 0.12:
            # Ưu tiên vùng có mật độ biên dày đặc và kích thước hợp lý
            diem_danh_gia = vung['so_diem'] * mat_do
            if diem_danh_gia > diem_tot_nhat:
                diem_tot_nhat = diem_danh_gia
                vung_bien_so = vung

    # Nếu ảnh chụp cận cảnh chỉ có mỗi biển số (chiếm trọn khung hình)
    if vung_bien_so is None:
        vung_bien_so = {'bbox': (0, 0, M - 1, N - 1)}

    x1, y1, x2, y2 = vung_bien_so['bbox']
    I_bien_so = I_xu_ly[y1:y2+1, x1:x2+1]

    # BƯỚC 4: Phân ngưỡng tách chữ số (vùng tối) bên trong biển số và tìm vùng liên thông
    # Dùng ngưỡng thích nghi theo phân vị độ sáng của vùng biển số (chữ số chiếm khoảng 25-35% diện tích tối nhất)
    nguong_chu = np.percentile(I_bien_so, 35)
    I_nhi_phan_chu = phan_nguong_muc_xam(I_bien_so, theta=nguong_chu, lay_vung_toi=True)

    cac_vung_chu = tim_vung_lien_thong(I_nhi_phan_chu, ban_kinh=1)
    nb_cao, nb_rong = I_bien_so.shape

    danh_sach_ky_tu = []
    for vung in cac_vung_chu:
        w, h = vung['rong'], vung['cao']
        ty_le_chu = w / float(h)
        # Chữ số có chiều cao từ 22% đến 92% chiều cao biển số, chiều ngang hẹp hơn chiều cao
        if (0.22 * nb_cao <= h <= 0.92 * nb_cao) and (0.02 * nb_rong <= w <= 0.35 * nb_rong) and (0.15 <= ty_le_chu <= 1.2):
            danh_sach_ky_tu.append(vung['bbox'])

    # Sắp xếp thứ tự ký tự từ dòng trên xuống dòng dưới, từ trái qua phải
    danh_sach_ky_tu.sort(key=lambda b: (b[1] // max(1, (nb_cao // 2)), b[0]))

    # Vẽ kết quả lên ảnh để hiển thị
    I_hien_thi_bs = cv2.cvtColor(I_xu_ly, cv2.COLOR_GRAY2RGB)
    cv2.rectangle(I_hien_thi_bs, (x1, y1), (x2, y2), (0, 255, 0), 2)

    I_hien_thi_ky_tu = cv2.cvtColor(I_bien_so, cv2.COLOR_GRAY2RGB)
    for (kx1, ky1, kx2, ky2) in danh_sach_ky_tu:
        cv2.rectangle(I_hien_thi_ky_tu, (kx1, ky1), (kx2, ky2), (255, 0, 0), 1)

    plt.figure(figsize=(16, 4))
    plt.suptitle(f"File: {duong_dan_anh} | Tách được {len(danh_sach_ky_tu)} vùng ký tự", fontsize=13, fontweight='bold')
    plt.subplot(1, 4, 1); plt.imshow(I_xu_ly, cmap='gray'); plt.title("1. Ảnh xám gốc"); plt.axis("off")
    plt.subplot(1, 4, 2); plt.imshow(I_nhi_phan_bien, cmap='gray'); plt.title("2. Biên Prewitt + Phân ngưỡng"); plt.axis("off")
    plt.subplot(1, 4, 3); plt.imshow(I_hien_thi_bs); plt.title("3. Khoanh vùng liên thông Biển số"); plt.axis("off")
    plt.subplot(1, 4, 4); plt.imshow(I_hien_thi_ky_tu); plt.title("4. Phân ngưỡng & Tách chữ số"); plt.axis("off")
    plt.tight_layout()
    plt.show()

# Chạy trực tiếp trên 3 file ảnh biển số xe trong thư mục /lab05/ của bạn
danh_sach_file_bsx = ["bsx1.jpg", "bsx2.jpeg", "bsx3.jpg"]
for ten_file in danh_sach_file_bsx:
    tach_bien_so_va_ky_tu_thuc_te(ten_file, ban_kinh_gom_bien=4)

In [ ]:
# Hàm hỗ trợ tạo 3 ảnh biển số xe giả lập thực tế (có thân xe, đèn xe, nhiễu nền và biển số)
def tao_3_anh_bien_so_mau():
    danh_sach_anh = []
    
    # Ảnh 1: Biển số ô tô 1 dòng nằm chính giữa
    img1 = np.random.randint(40, 70, (160, 320), dtype=np.uint8)
    cv2.rectangle(img1, (30, 20), (290, 140), 90, -1) # Thân xe
    cv2.rectangle(img1, (75, 65), (245, 115), 235, -1) # Nền biển số trắng
    cv2.rectangle(img1, (75, 65), (245, 115), 20, 2)   # Viền đen biển số
    cv2.putText(img1, "30A-12345", (83, 100), cv2.FONT_HERSHEY_SIMPLEX, 0.85, 15, 2)
    danh_sach_anh.append(("Ảnh 1: Biển dài 1 dòng", img1))

    # Ảnh 2: Biển số vuông 2 dòng nằm lệch phải
    img2 = np.random.randint(30, 60, (180, 320), dtype=np.uint8)
    cv2.rectangle(img2, (140, 45), (275, 140), 240, -1)
    cv2.rectangle(img2, (140, 45), (275, 140), 10, 2)
    cv2.putText(img2, "29-H1", (160, 85), cv2.FONT_HERSHEY_SIMPLEX, 0.9, 15, 2)
    cv2.putText(img2, "6789", (165, 128), cv2.FONT_HERSHEY_SIMPLEX, 0.9, 15, 2)
    danh_sach_anh.append(("Ảnh 2: Biển vuông 2 dòng", img2))

    # Ảnh 3: Biển số ô tô nằm lệch trái dưới thấp
    img3 = np.random.randint(50, 80, (160, 320), dtype=np.uint8)
    cv2.rectangle(img3, (25, 80), (195, 130), 230, -1)
    cv2.rectangle(img3, (25, 80), (195, 130), 20, 2)
    cv2.putText(img3, "99A-56789", (32, 115), cv2.FONT_HERSHEY_SIMPLEX, 0.85, 20, 2)
    danh_sach_anh.append(("Ảnh 3: Biển lệch góc dưới", img3))

    return danh_sach_anh

# Thực thi tách biển số và chữ số trên cả 3 ảnh
cac_anh_xe = tao_3_anh_bien_so_mau()

for ten_anh, I_xe in cac_anh_xe:
    I_bien, bbox_bs, I_crop_bs, ds_ky_tu = tach_bien_so_va_ky_tu(I_xe, nguong_bien=120, ban_kinh_gom_bien=3)
    
    # Vẽ khung phát hiện trên ảnh màu để hiển thị trực quan
    I_hien_thi_bs = cv2.cvtColor(I_xe, cv2.COLOR_GRAY2RGB)
    if bbox_bs:
        cv2.rectangle(I_hien_thi_bs, (bbox_bs[0], bbox_bs[1]), (bbox_bs[2], bbox_bs[3]), (0, 255, 0), 2)
        
    I_hien_thi_ky_tu = cv2.cvtColor(I_crop_bs, cv2.COLOR_GRAY2RGB) if I_crop_bs is not None else np.zeros((50, 100, 3), dtype=np.uint8)
    for (kx1, ky1, kx2, ky2) in ds_ky_tu:
        cv2.rectangle(I_hien_thi_ky_tu, (kx1, ky1), (kx2, ky2), (255, 0, 0), 1)

    # Hiển thị kết quả từng bước
    plt.figure(figsize=(15, 3.5))
    plt.suptitle(f"{ten_anh} (Tách được {len(ds_ky_tu)} ký tự)", fontsize=13, fontweight='bold')
    
    plt.subplot(1, 4, 1); plt.imshow(I_xe, cmap='gray'); plt.title("1. Ảnh xám ban đầu"); plt.axis("off")
    plt.subplot(1, 4, 2); plt.imshow(I_bien, cmap='gray'); plt.title("2. Biên Prewitt + Phân ngưỡng"); plt.axis("off")
    plt.subplot(1, 4, 3); plt.imshow(I_hien_thi_bs); plt.title("3. Vùng liên thông Biển số"); plt.axis("off")
    plt.subplot(1, 4, 4); plt.imshow(I_hien_thi_ky_tu); plt.title("4. Tách biển số & các chữ số"); plt.axis("off")
    plt.tight_layout()
    plt.show()